# 01b — Snapshot diff: 2026-09-09 vs the archive (~Apr 2025)

`data/raw/archive/` holds the boulder repo's raw scrape of the same site,
roughly seventeen months older. climbing-history.org is community-maintained
and actively edited, so the delta *is* data: which climbs were added, which
ascent counts grew, and — the interesting part — which grades moved between
snapshots, especially on milestone routes.

The archive was collected by the older scraper and still contains the header
defects, so both sides are pushed through `clean_detail_frame` before any
comparison (per `data/raw/archive/README.md`). Full row-level results are
written to `data/processed/snapshot_diff_*.csv`; only aggregates and the
notable rows are printed here.

In [1]:
import pandas as pd

from sportgradehistory import config
from sportgradehistory.clean import clean_detail_frame
from sportgradehistory.grades import clean_grade, font_ordinal, french_ordinal

new = pd.read_csv(config.CLEAN_DETAIL_CSV)
old = clean_detail_frame(pd.read_csv(config.ARCHIVE_DETAIL_CSV))
old["grade_clean"] = old["grade"].map(clean_grade)
new["grade_clean"] = new["grade"].map(clean_grade)

print(f"archive : {old.shape}   (~Apr 2025, boulder repo's raw scrape)")
print(f"current : {new.shape}   (2026-09-09)")

archive : (6844, 11)   (~Apr 2025, boulder repo's raw scrape)
current : (7669, 12)   (2026-09-09)


## Climbs added and removed

Joined on `climb_id`, which the site hands out sequentially and never reuses
(the id is in the URL).

In [2]:
merged = old.merge(new, on="climb_id", how="outer", suffixes=("_old", "_new"),
                   indicator=True)
added = merged[merged["_merge"] == "right_only"]
removed = merged[merged["_merge"] == "left_only"]
both = merged[merged["_merge"] == "both"]
print(f"in both snapshots : {len(both):,}")
print(f"added since       : {len(added):,}")
print(f"removed since     : {len(removed):,}\n")
print("added, by type:")
print(added["climb_type_new"].value_counts(dropna=False).head(8).to_string())
print("\nremoved, by type (pages deleted or now ascent-less):")
print(removed["climb_type_old"].value_counts(dropna=False).head(8).to_string())

in both snapshots : 6,840
added since       : 829
removed since     : 4

added, by type:
climb_type_new
Boulder problem             372
Sport route                 249
Trad climb                  143
Multi-pitch                  45
(approx) Boulder problem      5
Deep water solo               5
Aid Climb                     5
(approx) Trad climb           2

removed, by type (pages deleted or now ascent-less):
climb_type_old
Boulder problem        2
Sport route            1
(approx) Trad climb    1


## Ascents added

`num_ascents` counts the site's successful ascents; growth is the community
logging repeats (and occasionally pruning bad entries).

In [3]:
delta = (both["num_ascents_new"] - both["num_ascents_old"]).fillna(0).astype(int)
print(f"climbs whose ascent count grew : {(delta > 0).sum():,}")
print(f"total ascents added            : {delta[delta > 0].sum():,}")
print(f"climbs whose count shrank      : {(delta < 0).sum():,}\n")
gained = both.assign(gained=delta).nlargest(10, "gained")
print(gained[["climb_id", "climb_name_new", "grade_clean_new", "num_ascents_old",
              "num_ascents_new"]].to_string(index=False))

climbs whose ascent count grew : 612
total ascents added            : 1,096
climbs whose count shrank      : 11

 climb_id climb_name_new grade_clean_new  num_ascents_old  num_ascents_new
     2982   Hypergravity              8B                2             17.0
      154  The Hourglass              8B                8             22.0
     1714  The Globalist             8B+               11             22.0
     2762   The Valkyrie              8c                4             15.0
     1138    Hazel Grace             8B+               13             23.0
     4691    Draumkvedet              8c                1             10.0
       55    Illusionist              9a                9             17.0
      175 Mooiste Meisie              8B               17             25.0
     1469      City Park              E8                2             10.0
     2733     Muy Verdes              8c               11             19.0


## Grades that moved between snapshots

Compared on the *cleaned* grade string. These are live regrades the community
applied in the last seventeen months — a direct look at consensus in motion,
and exactly the mechanism that separates `as_proposed` from `as_consensus`.

In [4]:
moved = both[
    both["grade_clean_old"].notna() & both["grade_clean_new"].notna()
    & (both["grade_clean_old"] != both["grade_clean_new"])
].copy()

def direction(row):
    for ordinal in (french_ordinal, font_ordinal):
        a, b = ordinal(row["grade_clean_old"]), ordinal(row["grade_clean_new"])
        if a is not None and b is not None:
            return "upgrade" if b > a else "downgrade"
    return "rescale"  # moved between scales or off-scale; judged by hand

moved["direction"] = moved.apply(direction, axis=1)
out = moved[["climb_id", "climb_name_new", "climb_type_new", "grade_clean_old",
             "grade_clean_new", "direction", "num_ascents_new"]].rename(
    columns=lambda c: c.replace("_new", ""))
out.to_csv(config.PROCESSED_DIR / "snapshot_diff_grades_moved.csv", index=False)

print(f"{len(moved)} grades moved; by direction:")
print(moved["direction"].value_counts().to_string())
print("\nby type:")
print(moved["climb_type_new"].value_counts(dropna=False).head(6).to_string())
print(f"\nfull list -> data/processed/snapshot_diff_grades_moved.csv")

39 grades moved; by direction:
direction
downgrade    19
upgrade      11
rescale       9

by type:
climb_type_new
Boulder problem    18
Sport route        12
Trad climb          7
Multi-pitch         2

full list -> data/processed/snapshot_diff_grades_moved.csv


In [5]:
hard = moved[moved["grade_clean_new"].map(
    lambda g: (french_ordinal(g) or 0) >= french_ordinal("8c+")
    or (font_ordinal(g) or 0) >= font_ordinal("8B+"))]
cols = ["climb_id", "climb_name_new", "climb_type_new",
        "grade_clean_old", "grade_clean_new", "direction"]
print(f"the {len(hard)} moves at the hard end (sport 8c+ / Font 8B+ and up):")
print(hard[cols].to_string(index=False))

the 16 moves at the hard end (sport 8c+ / Font 8B+ and up):
 climb_id            climb_name_new  climb_type_new grade_clean_old grade_clean_new direction
       15                   Aladdin     Sport route              8c             8b+ downgrade
      708       Baa Baa Black Sheep     Sport route             8c+              8c downgrade
     2753                 Ego Death Boulder problem              8C             8B+ downgrade
     2849              Wind Up Bird     Sport route             9a+              9b   upgrade
     2919       Clash of the Titans     Sport route             9a+              9b   upgrade
     3261 Luftig kö till nålens öga Boulder problem              8B             8B+   upgrade
     4293                 Shantaram Boulder problem              8C             8B+ downgrade
     4526               Witchhammer     Sport route             8c+              9a   upgrade
     4989             Dying to Live Boulder problem             8B+              8C   upgrade


## Does any regrade touch a milestone route?

Checked against the milestone tables under **both** conventions plus the two
disputed claims — a regrade here would mean the milestone story itself moved
between snapshots.

In [6]:
from sportgradehistory.milestones import (
    DISPUTED_CLAIMS, load_sport, milestone_table)

sport = load_sport()
watched = set(DISPUTED_CLAIMS)
for convention in ("as_proposed", "as_consensus"):
    watched |= set(milestone_table(sport, convention)["climb_id"].astype(int))

touched = moved[moved["climb_id"].isin(watched)]
if touched.empty:
    print(f"none of the {len(watched)} milestone/disputed climb_ids moved grade "
          "between snapshots")
else:
    print("MILESTONE ROUTES WHOSE GRADE MOVED:")
    print(touched[cols].to_string(index=False))

none of the 15 milestone/disputed climb_ids moved grade between snapshots


## New entrants at the top end

Routes in the fresh snapshot, absent from the archive, graded 9a or harder —
the seventeen months of new hard-route history this scrape captures.

In [7]:
new_hard = added[added["grade_clean_new"].map(
    lambda g: (french_ordinal(g) or -1) >= french_ordinal("9a"))]
show = new_hard[["climb_id", "climb_name_new", "grade_clean_new",
                 "first_climber_new", "first_ascent_date_new"]].rename(
    columns=lambda c: c.replace("_new", ""))
show.to_csv(config.PROCESSED_DIR / "snapshot_diff_new_9a_plus.csv", index=False)
print(f"{len(new_hard)} new 9a+ entries -> data/processed/snapshot_diff_new_9a_plus.csv")
print(show.sort_values("climb_id").head(15).to_string(index=False))

69 new 9a+ entries -> data/processed/snapshot_diff_new_9a_plus.csv
 climb_id           climb_name grade_clean            first_climber first_ascent_date
     3111           El Picacho          9b             Jonatan Flor              2021
     3957     Priorato de Sion          9a Ramón Julián Puigblanque     24th Mar 2008
     4623                Hello          9a               Adam Ondra     17th Sep 2017
     7514         Nid de Fadas          9a              Pepa Šindel     27th Feb 2025
     7539     Blood of Olympus          9a              Josh Cornah              2018
     7540             Superboy          9a           Derek Thatcher              2019
     7541       Phantom Bender          9a           Derek Thatcher              2019
     7566            Frostbite          9a            Ryan Sklenica      5th Apr 2026
     7575              Juturna          9a           Jorg Verhoeven     19th Aug 2020
     7590            Biologico          9a               Adam Ondra      